I wanted a way to relate schemas to their table names. I know this is probably not a good idea, but I was too lazy to properly do this :P.

In [ ]:
import importlib
import inspect
from types import ModuleType
from typing import List
from pandera.pandas import DataFrameModel
import transformations # Replace with your actual package name
import os


def get_all_schema_models(package: ModuleType) -> List[DataFrameModel]:
    schema_classes = []
    root_path = package.__path__[0]
    base_package = package.__name__

    for dirpath, _, filenames in os.walk(root_path):
        for filename in filenames:
            if filename.endswith(".py") and not filename.startswith("__"):
                file_path = os.path.join(dirpath, filename)
                rel_path = os.path.relpath(file_path, root_path)
                module_name = os.path.splitext(rel_path)[0].replace(os.sep, ".")
                full_module_name = f"{base_package}.{module_name}"

                try:
                    module = importlib.import_module(full_module_name)
                    for name, obj in inspect.getmembers(module, inspect.isclass):
                        if issubclass(obj, DataFrameModel) and obj is not DataFrameModel:
                            schema_classes.append(obj)
                except Exception:
                    continue  # Skip modules that fail to import

    return schema_classes

schemas = get_all_schema_models(transformations)
schemas

In [ ]:
import re 

def to_snake_case(name):
    return re.sub(r'(?<!^)(?=[A-Z])', '_', name).lower()

schema_dict = {
    to_snake_case(schema.__name__): schema
    for schema in schemas
}
schema_dict

In [ ]:
import sqlite3
import pandas as pd
import pandera as pa
import logging
from transformations.utils.io import get_metadata_from_schema
import json
from pathlib import Path
logger = logging.getLogger()

def to_snake_case(name):
    return re.sub(r'(?<!^)(?=[A-Z])', '_', name).lower()



def write_sqlite_db_to_csvs(db_path: str, output_dir: str, schemas: list[pa.DataFrameModel]) -> str:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    schema_dict = {
        to_snake_case(schema.__name__): schema
        for schema in schemas
    }

    # Get all table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    for table_name in tables:
        table_name = table_name[0]
        df = pd.read_sql(f"select * from {table_name};", con=conn)
        try:
            df_schema = schema_dict[table_name]
            metadata = get_metadata_from_schema(df_schema, df)
            metadata_output_dir = output_dir + "/metadata"
            Path(metadata_output_dir).mkdir(parents=True, exist_ok=True)
            with open(metadata_output_dir + f"/{table_name}.json", "w") as f:
                json.dump(metadata, f, indent=4)
            df.to_csv(f"{output_dir}/{table_name}.csv", index=False)
        except:
            logger.exception(f"Couldn't generate metadata for {table_name}")
            

conn = sqlite3.connect("../data/gold/gold.db")